# LLM profile and knowledge prompts

Run from the project root. Previous versions are preserved in archive/. Saved inputs remain in outputs/; results are written to analysis_results/ and figures/. Run llm_outputs_analysis.ipynb, icpsr.ipynb, and then tables.ipynb. Model generation is disabled by default.


In [2]:
import json
import os
import pandas as pd
import numpy as np
import pprint
from sklearn.metrics.pairwise import cosine_distances
from pathlib import Path
from analysis_utils import PAPER_PAIRS, MODELS, DATA_DIR, RESULT_DIR, FIGURE_DIR, normalize_knowledge
RUN_GENERATION = False
GENERATION_DIR = Path("generated_outputs")
GENERATION_DIR.mkdir(exist_ok=True)
MODEL_IDS = {"claudesonnet4.6": "us.anthropic.claude-sonnet-4-6", "deepseek.v3.2": "deepseek.v3.2", "gpt-oss-120b": "openai.gpt-oss-120b-1:0"}


In [3]:
file = '2018-09-13--06.02.24.events.csv'
sim_data = pd.read_csv(file, header=4)

sim_data["HCV"] = sim_data["HCV"].astype("string").str.strip().str.lower().replace({"infectiousacute":"acute"})

sim_data["Info"] = sim_data.get("Info", pd.Series()).astype(str)
sim_data = sim_data.drop(index=[31330, 4816], errors='ignore')

drop_events = ['vaccinated', 'exposed']
clean_sim_data = sim_data[~sim_data['HCV'].isin(drop_events)].copy()

print("HCV Counts (Cleaned)")
print(clean_sim_data['HCV'].value_counts())

agent_histories = {
    agent_id: group.sort_values('Time')
    for agent_id, group in clean_sim_data.groupby('Agent')}


llm_prompts.ipynb:cell3:2: DtypeWarning: Columns (0: Info) have mixed types. Specify dtype option on import or set low_memory=False.
HCV Counts (Cleaned)
HCV
susceptible    29700
chronic        14627
recovered      12032
acute           6670
Name: count, dtype: Int64


In [4]:
baseline = (
    clean_sim_data
    .sort_values(['Time']) 
    .drop_duplicates(subset='Agent', keep='first')
)
baseline = baseline[baseline['HCV'] == 'susceptible'].copy()
baseline['Injection_Experience'] = baseline['Age'] - baseline['Age_Started']

event_counts = clean_sim_data.groupby('Agent')['Time'].nunique()
valid_agents = event_counts[event_counts >= 2].index

baseline = baseline[
    (baseline['Age'] >= 18) &
    (baseline['Daily_injection_intensity'] > 0) &
    (baseline['Agent'].isin(valid_agents))
].copy()

# split by experience
young_eligible = baseline[baseline['Injection_Experience'] < 3].copy()
young_eligible['Age_Group'] = 'Young'

old_eligible = baseline[
    baseline['Injection_Experience'] >= 3
].copy()
old_eligible['Age_Group'] = 'Old'

all_eligible = pd.concat([young_eligible, old_eligible])

N_PER_BUCKET = 7
STRATA_COLS = ['Gender', 'Race', 'Age_Group']

# first stratified sample
stratified_agents = pd.concat([group.sample(7, random_state=42) for _, group in all_eligible.groupby(STRATA_COLS, sort=True)])

used_agents = set(stratified_agents['Agent'])
remaining_pool = all_eligible[~all_eligible['Agent'].isin(used_agents)].copy()

# second stratified sample
extra_agents = pd.concat([group.sample(n=min(7, len(group)), random_state=99) for _, group in remaining_pool.groupby(STRATA_COLS, sort=True)])
final_sample = pd.concat([stratified_agents, extra_agents])
assert final_sample.Agent.nunique() == 224
assert final_sample.groupby(STRATA_COLS).size().eq(14).all()


In [5]:
data = final_sample.copy()
behav_map = {
    "Daily_injection_intensity": "Weekly injections",
    "Fraction_recept_sharing": "Receptive sharing",
    "Drug_in_degree": "Netowork in-degree",
    "Drug_out_degree": "Network out-degree",
    "HCV_friend_preval": "Partners with HCV"}

group_vars = ["Race", "Age_Group", "Gender"]
all_results = []
df_z = data.copy()

for col in behav_map.keys():
    if col == "Daily_injection_intensity":
        df_z[col] = df_z[col] * 7

for col in behav_map.keys():
    df_z[col] = (df_z[col] - df_z[col].mean()) / df_z[col].std()

for group in group_vars:
    for col, label in behav_map.items():

        group_means = df_z.groupby(group)[col].mean()

        all_results.append({
            "Group": group,
            "Variable": label,
            "SD_between_group_means": group_means.std(),
            "Max Group": group_means.idxmax(),
            "Min Group": group_means.idxmin()})

race_table = pd.DataFrame(all_results)
print(race_table)

        Group            Variable  SD_between_group_means Max Group Min Group
0        Race   Weekly injections                0.221314     Other   NHWhite
1        Race   Receptive sharing                0.209826   NHWhite   NHBlack
2        Race  Netowork in-degree                0.174461   NHWhite   NHBlack
3        Race  Network out-degree                0.171934   NHWhite   NHBlack
4        Race   Partners with HCV                0.107086   NHBlack     Other
5   Age_Group   Weekly injections                0.055742     Young       Old
6   Age_Group   Receptive sharing                0.017532       Old     Young
7   Age_Group  Netowork in-degree                0.000000       Old       Old
8   Age_Group  Network out-degree                0.073267     Young       Old
9   Age_Group   Partners with HCV                0.186618       Old     Young
10     Gender   Weekly injections                0.008478    Female      Male
11     Gender   Receptive sharing                0.098561    Fem

In [6]:
print(final_sample['HCV'].value_counts())
print(final_sample['Agent'].duplicated().sum())

agent_ids = final_sample['Agent'].unique()
agent_ids_list = final_sample['Agent'].tolist()

print(f"Number of unique IDs: {len(agent_ids)}")
print(agent_ids[:10])

HCV
susceptible    224
Name: count, dtype: Int64
0
Number of unique IDs: 224
[ 490771391  208393663 1760779551 1110995944 2018099683  700138803
 1238160207 1775692649 2009710957 1430711947]


In [7]:
prompt_list = []
for agent_id in agent_ids[:]:
    agent_df = agent_histories[agent_id]
    row = agent_df[agent_df['Event'] == 'activated'].iloc[0].copy()

    race_map = {
    "NHBlack": "Black",
    "NHWhite": "White",
    "Hispanic": "Hispanic",
    "Other": "a race other than Black, White, or Hispanic"
}
   
    cleaned_race = race_map.get(row['Race'])
    
    prompt = (
        "Chicago drug environment: "
        "Homelessness is defined as lacking a stable nighttime residence, including unstable or non-permanent living situations. "
        "Public settings include streets, parks, shelters, and Chicago Transit Authority stations. "
        "On-site and mobile harm reduction services provide clean syringes, naloxone, fentanyl test strips, and opioid use disorder treatment, which can be accessed by walking, car, or public transportation.\n\n"

        "Individual profile prior to the scenario onset:\n"
        f"- Age: {round(row['Age'])}\n"
        f"- Gender: {row['Gender']}\n"
        f"- Race: {cleaned_race}\n"
        "- Hepatitis C virus (HCV) status: susceptible\n"
        "- Housing status: stable for at least a year\n" #homeless for at least a month 
        f"- Injection history: Started at age {round(row['Age_Started'])}\n"
        f"- Injection frequency: ~{round(row['Daily_injection_intensity']*7)} injections per week; ~{int(row['Fraction_recept_sharing']*100)}% involved reusing someone else's syringe\n"
        f"- Network: Receives used syringes from {round(row['Drug_in_degree'])} persons and gives used syringes to {round(row['Drug_out_degree'])} persons; {int(row['HCV_friend_preval']*100)}% HCV prevalence among current injection partners\n\n"
        
        "Scenario: During a 6-month period in 2026, this individual is highly confident that illicitly manufactured fentanyl is present in their opioid supply.\n"
        "Instructions: "
        "Predict the individual's typical weekly behavior during the last 4 weeks of this scenario. Report measures 1–6 as a weekly average, measures 7-9 as a proportion (between 0 and 1 inclusive, rounded to 2 decimal places), and measures 10-12 as categorical values:\n\n"
        
        "1. Number of injections\n"
        "2. Number of unique persons they gave a used syringe to \n"
        "3. Number of unique persons they received a used syringe from \n"
        "4. Number of unique injection partners (someone they have injected with at least once in the past 4 weeks)\n"
        "5. Number of times they chose to smoke or snort drugs instead of injecting\n" 
        "6. Number of times they experienced a non-fatal overdose\n" 
        "7. Proportion of injections where they reused a syringe that someone else already used\n"
        "8. Proportion of injection partners currently infected with HCV\n"
        "9. Proportion of injections done in a public setting\n"
        "10. Predominant neighborhood of drug-use activity during the last 4 weeks\n"
        "11. Enrolled in an opioid use disorder treatment program during the last 4 weeks\n"
        "12. Current HCV infection state (fixed true state, regardless of individual awareness)\n\n"

        "Output Format: "
        "Return answers as a single-line JSON object with the following keys. Output ONLY the JSON object. No explanation, no markdown, no code blocks, no text before or after.\n\n"
        "Injections\n"
        "Drug_out\n"
        "Drug_in\n"
        "Partners\n"
        "Alt_route\n"
        "Overdose\n"
        "Recept_sharing\n"
        "Partner_HCV_prop\n"
        "Public_prop\n"
        "Neighborhood (north side, central, west side, or south side)\n"
        "Treatment (0=no, 1=yes)\n"
        "HCV (susceptible or acute)"
        )

    
    prompt2 = (
        "Chicago drug landscape: "
        "Homelessness is defined as lacking a stable nighttime residence, including unstable or temporary living situations. "
        "Public settings include streets, parks, shelters, and Chicago Transit Authority stations. "
        "On-site and mobile harm reduction programs provide clean syringes, naloxone, fentanyl test strips, and opioid use disorder treatment, which can be accessed by walking, car, or public transit.\n\n"
        
        "Individual profile prior to the scenario onset:\n"
        f"- Age: {round(row['Age'])}\n"
        f"- Gender: {row['Gender']}\n"
        f"- Race: {cleaned_race}\n"
        "- Hepatitis C virus (HCV) status: susceptible\n"
        "- Housing status: stable for at least a year\n" #homeless for at least a month
        f"- Injection history: Started at age {round(row['Age_Started'])}\n"
        f"- Injection frequency: ~{round(row['Daily_injection_intensity']*7)} injections per week; ~{int(row['Fraction_recept_sharing']*100)}% involved reusing someone else's syringe\n"
        f"- Network: Receives used syringes from {round(row['Drug_in_degree'])} persons and gives used syringes to {round(row['Drug_out_degree'])} persons; {int(row['HCV_friend_preval']*100)}% HCV prevalence among current injection partners\n\n"
        
        "Scenario: During a 6-month period in 2026, this individual is highly confident that illicitly manufactured fentanyl is present in their opioid supply.\n"
        "Instructions: "
        "Predict the individual's typical weekly behavior during the last 4 weeks of this scenario. Report measures 1–6 as a weekly average, measures 7-9 as a proportion (between 0 and 1 inclusive, rounded to 2 decimal places), and measures 10-12 as categorical values:\n\n"
        
        "1. Number of injections\n"
        "2. Number of unique persons they gave a used syringe to \n"
        "3. Number of unique persons they received a used syringe from \n"
        "4. Number of unique injection partners (someone they have injected with at least once in the past 4 weeks)\n"
        "5. Number of times they chose to smoke or snort drugs instead of injecting\n" 
        "6. Number of times they experienced a non-fatal overdose\n" 
        "7. Proportion of injections where they reused a syringe that someone else already used\n"
        "8. Proportion of injection partners currently infected with HCV\n"
        "9. Proportion of injections done in a public setting\n"
        "10. Predominant neighborhood of drug-use activity during the last 4 weeks\n"
        "11. Enrolled in an opioid use disorder treatment program during the last 4 weeks\n"
        "12. Current HCV infection state (fixed true state, regardless of individual awareness)\n\n"

        "Output Format: "
        "Return answers as a single-line JSON object with the following keys. Output ONLY the JSON object. No explanation, no markdown, no code blocks, no text before or after.\n\n"
        "Injections\n"
        "Drug_out\n"
        "Drug_in\n"
        "Partners\n"
        "Alt_route\n"
        "Overdose\n"
        "Recept_sharing\n"
        "Partner_HCV_prop\n"
        "Public_prop\n"
        "Neighborhood (north side, central, west side, or south side)\n"
        "Treatment (0=no, 1=yes)\n"
        "HCV (susceptible or acute)"
    )
    
    prompt3 = (
        "Chicago drug injection landscape: "
        "Homelessness refers to not having a consistent place to sleep at night, including temporary and volatile housing arrangements. "
        "Public locations include outdoor areas, streets, parks, shelters, and Chicago Transit Authority stations. "
        "Harm reduction resources available through fixed-site and mobile programs include sterile syringes, naloxone, fentanyl test strips, and medications for opioid use disorder. These services may be reached by walking, driving, or using public transit.\n\n"

        "Individual characteristics before the start of the scenario:\n"
        f"- Age: {round(row['Age'])}\n"
        f"- Gender: {row['Gender']}\n"
        f"- Race/ethnicity: {cleaned_race}\n"
        "- Hepatitis C virus (HCV) status: susceptible\n"
        "- Housing status: Living in stable housing for at least one year\n" #Experiencing homelessness for at least one month
        f"- Injection history: Began injecting at age {round(row['Age_Started'])}\n"
        f"- Injection frequency: Approximately {round(row['Daily_injection_intensity']*7)} injections per week; approximately {int(row['Fraction_recept_sharing']*100)}% involved using a syringe previously used by another person\n"
        f"- Injection network: Obtains used syringes from {round(row['Drug_in_degree'])} individuals and provides used syringes to {round(row['Drug_out_degree'])} individuals; {int(row['HCV_friend_preval']*100)}% of current injection partners have HCV\n\n"

        "Scenario: Over a 6-month period in 2026, this individual is highly certain that illicitly manufactured fentanyl exists in their opioid supply.\n"
        "Instructions: "
        "Estimate the individual's typical weekly behaviors during the final 4 weeks of this scenario. Provide items 1–6 as weekly averages, items 7–9 as proportions from 0 to 1 rounded to two decimal places, and items 10–12 as categorical responses:\n\n"

        "1. Total number of injection events\n"
        "2. Number of different people they gave a used syringe to\n"
        "3. Number of different people they received a used syringe from\n"
        "4. Number of distinct injection partners (defined as someone they injected with at least once in the previous 4 weeks)\n"
        "5. Number of occasions they used an alternative injection route (smoking or snorting)\n"
        "6. Number of non-fatal overdose events experienced\n"
        "7. Fraction of injections involving reuse of a syringe previously used by another person\n"
        "8. Fraction of current injection partners estimated to have HCV infection\n"
        "9. Fraction of injections occurring in public locations\n"
        "10. Main neighborhood where drug use occurred during the previous 4 weeks\n"
        "11. Whether the individual was enrolled in an opioid use disorder treatment program during the last 4 weeks\n"
        "12. Current HCV infection status (use the true assigned status regardless of whether the individual knows it)\n\n"

        "Output Format: "
        "Provide the response as exactly one single-line JSON object containing the following keys. Output ONLY the JSON object; do not include explanations, markdown, code blocks, or additional text.\n\n"
        "Injections\n"
        "Drug_out\n"
        "Drug_in\n"
        "Partners\n"
        "Alt_route\n"
        "Overdose\n"
        "Recept_sharing\n"
        "Partner_HCV_prop\n"
        "Public_prop\n"
        "Neighborhood (north side, central, west side, or south side)\n"
        "Treatment (0=no, 1=yes)\n"
        "HCV (susceptible or acute)"
        )

    for paraphrase, wording in enumerate([prompt, prompt2, prompt3]):
        for homeless in [0, 1]:
            text = wording
            if homeless:
                text = text.replace("- Housing status: stable for at least a year",
                                    "- Housing status: homeless for at least a month")
                text = text.replace("- Housing status: Living in stable housing for at least one year",
                                    "- Housing status: Experiencing homelessness for at least one month")
            prompt_list.append({"Agent": int(agent_id), "Paraphrase": paraphrase,
                                "Homeless": homeless, "prompt": text})  

assert len(prompt_list) == 224 * 2 * 3
print(prompt_list[0]["prompt"])

Chicago drug environment: Homelessness is defined as lacking a stable nighttime residence, including unstable or non-permanent living situations. Public settings include streets, parks, shelters, and Chicago Transit Authority stations. On-site and mobile harm reduction services provide clean syringes, naloxone, fentanyl test strips, and opioid use disorder treatment, which can be accessed by walking, car, or public transportation.

Individual profile prior to the scenario onset:
- Age: 48
- Gender: Female
- Race: Hispanic
- Hepatitis C virus (HCV) status: susceptible
- Housing status: stable for at least a year
- Injection history: Started at age 44
- Injection frequency: ~29 injections per week; ~5% involved reusing someone else's syringe
- Network: Receives used syringes from 1 persons and gives used syringes to 0 persons; 0% HCV prevalence among current injection partners

Scenario: During a 6-month period in 2026, this individual is highly confident that illicitly manufactured fent

In [8]:
pairs = list(PAPER_PAIRS)
assert len(pairs) == 32 and not any("fentanyl" in " ".join(pair).lower() for pair in pairs)


In [9]:
prompts = []

for exposure, outcome in pairs:
    prompt3 = f"""
You are an epidemiologic evidence extraction system.

Question:
Based on epidemiologic studies of people who inject drugs (PWID), what is the expected relationship between {exposure} and {outcome}?

Return ONLY a valid JSON object in this format:

{{
  "exposure": "{exposure}",
  "outcome": "{outcome}",
  "direction": "increase, decrease, mixed, or uncertain",
  "evidence_quality": "weak, moderate, strong, or uncertain",
  "estimate": {{
    "measure": "aOR, aRR, aPR, OR, PR, RR, HR, or null",
    "value": "point estimate or null",
    "lower_ci": "lower 95% CI or null",
    "upper_ci": "upper 95% CI or null"
  }}
}}

Rules:
1. Use PWID epidemiologic literature only.
2. Prefer adjusted estimates (aOR, aRR, aPR, aHR) when available.
3. If multiple estimates exist, prefer pooled estimates from meta-analyses; if none are available, use the adjusted estimate from the largest or most recent cohort study.
4. If no quantitative estimate is explicitly reported, set all fields in "estimate" to "null".
5. Do NOT infer, approximate, or hallucinate numerical values or confidence intervals.
6. Direction reflects consistency across studies:
   - increase = mostly positive associations
   - decrease = mostly negative associations
   - mixed = substantial inconsistency across studies
   - uncertain = insufficient evidence
7. evidence_quality:
   - weak = few studies or inconsistent findings
   - moderate = some consistent cohort evidence
   - strong = meta-analysis or consistent cohort evidence across multiple studies
8. Do not include any text outside the JSON object.
"""
    prompts.append(prompt3)

print(prompts[0])


You are an epidemiologic evidence extraction system.

Question:
Based on epidemiologic studies of people who inject drugs (PWID), what is the expected relationship between current or recent homelessness and public injecting?

Return ONLY a valid JSON object in this format:

{
  "exposure": "current or recent homelessness",
  "outcome": "public injecting",
  "direction": "increase, decrease, mixed, or uncertain",
  "evidence_quality": "weak, moderate, strong, or uncertain",
  "estimate": {
    "measure": "aOR, aRR, aPR, OR, PR, RR, HR, or null",
    "value": "point estimate or null",
    "lower_ci": "lower 95% CI or null",
    "upper_ci": "upper 95% CI or null"
  }
}

Rules:
1. Use PWID epidemiologic literature only.
2. Prefer adjusted estimates (aOR, aRR, aPR, aHR) when available.
3. If multiple estimates exist, prefer pooled estimates from meta-analyses; if none are available, use the adjusted estimate from the largest or most recent cohort study.
4. If no quantitative estimate is ex

In [10]:
prompts2 = []

for factor1, factor2 in pairs:
    prompt2 = f"""
You are an epidemiological evidence assessment system.

Question:
For people who inject drugs (PWID), what does the available epidemiological evidence suggest about the association of {factor1} with {factor2}?

Return ONLY a valid JSON object using the format below:

{{
  "factor1": "{factor1}",
  "factor2": "{factor2}",
  "direction": "increase, decrease, mixed, or uncertain",
  "evidence_quality": "weak, moderate, strong, or uncertain",
  "estimate": {{
    "measure": "aOR, aRR, aPR, OR, PR, RR, HR, or null",
    "value": "point estimate or null",
    "lower_ci": "lower 95% CI or null",
    "upper_ci": "upper 95% CI or null"
  }}
}}

Rules:
1. Only use evidence from epidemiological studies of PWID.
2. Use an adjusted measure of association if available.
3. If several quantitative results are available, use a pooled estimate. If no pooled estimate is available, use an adjusted estimate from the largest or most recent cohort study.
4. Report quantitative results only when they are explicitly given in the literature. Otherwise, set all fields under "estimate" to "null".
5. Do NOT calculate, infer, approximate, or hallucinate numerical estimates or confidence intervals.
6. Assign the direction according to the predominant evidence:
   - increase = predominantly positive associations
   - decrease = predominantly negative associations
   - mixed = substantial inconsistency across studies
   - uncertain = insufficient evidence to determine the direction
7. Assign evidence quality according to:
   - weak = few studies
   - moderate = some consistent cohort evidence
   - strong = meta-analysis or consistent cohort evidence across multiple studies
8. Return no text other than the JSON object.
"""
    prompts2.append(prompt2)

print(prompts2[0])


You are an epidemiological evidence assessment system.

Question:
For people who inject drugs (PWID), what does the available epidemiological evidence suggest about the association of current or recent homelessness with public injecting?

Return ONLY a valid JSON object using the format below:

{
  "factor1": "current or recent homelessness",
  "factor2": "public injecting",
  "direction": "increase, decrease, mixed, or uncertain",
  "evidence_quality": "weak, moderate, strong, or uncertain",
  "estimate": {
    "measure": "aOR, aRR, aPR, OR, PR, RR, HR, or null",
    "value": "point estimate or null",
    "lower_ci": "lower 95% CI or null",
    "upper_ci": "upper 95% CI or null"
  }
}

Rules:
1. Only use evidence from epidemiological studies of PWID.
2. Use an adjusted measure of association if available.
3. If several quantitative results are available, use a pooled estimate. If no pooled estimate is available, use an adjusted estimate from the largest or most recent cohort study.
4

In [11]:
try:
    import tiktoken
except ImportError:
    print("Optional token diagnostic skipped: tiktoken is unavailable.")
else:
    context = (
        "Chicago drug environment: "
        "Homelessness is defined as lacking a stable nighttime residence, including unstable or non-permanent living situations. "
        "Public settings include streets, parks, shelters, and Chicago Transit Authority stations. "
        "On-site and mobile harm reduction services provide clean syringes, naloxone, fentanyl test strips, and opioid use disorder treatment, which can be accessed by walking, car, or public transportation."
    )
    enc = tiktoken.get_encoding("cl100k_base")  # GPT-4 tokenizer as approximation
    
    housing_homeless = "homeless for at least a month"
    housing_stable = "stable for at least a year"
    
    print(len(enc.encode(housing_homeless)))  # count tokens changed
    print(len(enc.encode(housing_stable)))
    print(len(enc.encode(context)))


Optional token diagnostic skipped: tiktoken is unavailable.


In [12]:
try:
    import tiktoken
except ImportError:
    print("Optional token diagnostic skipped: tiktoken is unavailable.")
else:
    context2 = (
            "Chicago drug landscape: "
            "Homelessness is defined as lacking a stable nighttime residence, including unstable or temporary living situations. "
            "Public settings include streets, parks, shelters, and Chicago Transit Authority stations. "
            "On-site and mobile harm reduction programs provide clean syringes, naloxone, fentanyl test strips, and opioid use disorder treatment, which can be accessed by walking, car, or public transit."
    )
    print(len(enc.encode(context2)))
    
    old_words = ["environment", "transportation", "non-permanent", "services", "situations"]
    new_words = ["landscape", "transit", "temporary", "programs", "conditions"]
    
    print("old:", sum(len(enc.encode(w)) for w in old_words))
    print("new:", sum(len(enc.encode(w)) for w in new_words))


Optional token diagnostic skipped: tiktoken is unavailable.


In [13]:
# Optional knowledge generation. No API calls occur by default.
responses_list = []
if RUN_GENERATION:
    import boto3
    client = boto3.client(service_name="bedrock-runtime", region_name="us-east-1")
    destination = GENERATION_DIR / "knowledge_outputs.jsonl"
    existing = set()
    if destination.exists():
        for line in destination.read_text(encoding="utf-8").splitlines():
            entry = json.loads(line)
            existing.add((entry["Model"], entry["exposure"], entry["outcome"], entry["Paraphrase"]))
    for model, api_id in MODEL_IDS.items():
        for paraphrase, wording_list in enumerate([prompts, prompts2]):
            for (exposure, outcome), api_text in zip(pairs, wording_list):
                key = (model, exposure, outcome, paraphrase)
                if key in existing:
                    continue
                result = client.converse(modelId=api_id, messages=[{"role":"user", "content":[{"text":api_text}]}], inferenceConfig={"temperature":0})
                text = " ".join(b["text"] for b in result["output"]["message"]["content"] if "text" in b)
                response = json.loads(text.replace("```json", "").replace("```", "").strip())
                if not isinstance(response, dict):
                    raise ValueError("Knowledge response must be a JSON object.")
                entry = dict(Model=model, Paraphrase=paraphrase, exposure=exposure, outcome=outcome, response=response)
                with destination.open("a", encoding="utf-8") as f:
                    f.write(json.dumps(entry) + "\n")
                existing.add(key)
                responses_list.append(entry)
else:
    print("Knowledge generation skipped; use saved outputs.")


Knowledge generation skipped; use saved outputs.


In [14]:
# Optional profile generation, explicitly indexed by agent, wording, and housing.
responses_list = []
if RUN_GENERATION:
    import boto3
    client = boto3.client(service_name="bedrock-runtime", region_name="us-east-1")
    destination = GENERATION_DIR / "outputs.jsonl"
    existing = set()
    if destination.exists():
        for line in destination.read_text(encoding="utf-8").splitlines():
            e = json.loads(line)
            existing.add((e["Model"], e["Agent"], e["Paraphrase"], e["Homeless"]))
    for model, api_id in MODEL_IDS.items():
        for prompt_entry in prompt_list:
            key = (model, prompt_entry["Agent"], prompt_entry["Paraphrase"], prompt_entry["Homeless"])
            if key in existing:
                continue
            result = client.converse(modelId=api_id, messages=[{"role":"user", "content":[{"text":prompt_entry["prompt"]}]}], inferenceConfig={"temperature":0})
            text = " ".join(b["text"] for b in result["output"]["message"]["content"] if "text" in b)
            response = json.loads(text.replace("```json", "").replace("```", "").strip())
            if not isinstance(response, dict) or response.get("HCV") not in ["susceptible", "acute"]:
                raise ValueError("Profile response must be a JSON object with susceptible/acute HCV status.")
            entry = {k:v for k,v in prompt_entry.items() if k != "prompt"}
            entry.update(Model=model, response=response)
            with destination.open("a", encoding="utf-8") as f:
                f.write(json.dumps(entry) + "\n")
            existing.add(key)
            responses_list.append(entry)
else:
    print("Profile generation skipped; use saved outputs.")


Profile generation skipped; use saved outputs.


for index, api_text in enumerate(prompt_list): 
    current_agent_id = id_batches[0][index]
    print(current_agent_id)
    print(type(current_agent_id.item()))

In [16]:
# Explicit JSON-object validation; no missing intermediate cleaning file.
def flatten_generated(path):
    rows = []
    for line in path.read_text(encoding="utf-8").splitlines():
        entry = json.loads(line)
        response = entry.pop("response")
        if isinstance(response, str):
            response = json.loads(response.replace("```json", "").replace("```", "").strip())
        if not isinstance(response, dict):
            raise ValueError("Generated response is not a JSON object.")
        estimate = response.pop("estimate", {})
        if isinstance(estimate, dict):
            response.update(estimate)
        for identifier in ["Model", "Agent", "Paraphrase", "Homeless", "exposure", "outcome"]:
            if identifier in response and identifier in entry and response[identifier] != entry[identifier]:
                raise ValueError(f"Response conflicts with metadata: {identifier}")
        rows.append({**response, **entry})
    return pd.DataFrame(rows)


In [17]:
if RUN_GENERATION:
    generated = flatten_generated(GENERATION_DIR / "outputs.jsonl")
    if generated.duplicated(["Model", "Agent", "Paraphrase", "Homeless"]).any():
        raise ValueError("Duplicate generated profile identifiers.")
    generated.to_csv(GENERATION_DIR / "outputs.csv", index=False)


In [18]:
if RUN_GENERATION:
    generated = normalize_knowledge(flatten_generated(GENERATION_DIR / "knowledge_outputs.jsonl"))
    generated.to_csv(GENERATION_DIR / "knowledge_outputs.csv", index=False)
